# Chapter 13 Companion Notebook: Dimension Reduction with PCA

**Book:** *Business Analytics and Artificial Intelligence: An Advanced Guide to Data-Driven Decision Making*  
**Book authors:** Hyunhwan "Aiden" Lee and Reo Song

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/indy16mm/business-analytics-ai/blob/main/notebooks/Ch13_Dimension_Reduction_Wine_PCA.ipynb)

This notebook accompanies Chapter 13 of the book.

**License:** Use of this notebook is governed by the repository's
[Limited Companion Materials License](../LICENSE).



# Wine: Principle component analysis
https://archive.ics.uci.edu/dataset/109/wine
- Class: The type of wine (1, 2, or 3), representing three different grape varieties.  
- Alcohol: The alcohol content of the wine (%).  
- Malic_acid: The amount of malic acid in the wine, affecting its tartness.  
- Ash: The total ash content in wine (g/100ml), which influences mineral content.  
- Alcalinity_of_ash: The alkalinity of the ash (pH), which relates to the wine’s acidity balance.  
- Magnesium: Magnesium concentration (mg/l), an essential mineral affecting wine stability.  
- Phenols: The total phenolic content, contributing to the wine’s color, taste, and mouthfeel.  
- Flavanoids: A subclass of phenols, flavonoids impact bitterness, astringency, and antioxidant properties.  
- Nonflavanoid_phenols: Phenols not classified as flavonoids, affecting the color and aging potential of wine.  
- Proanthocyanins: Another class of phenols, contributing to color stability and bitterness.  
- Color_intensity: The intensity of the wine's color.  
- Hue: The hue of the wine (red to yellow ratio), describing color variation.
- OD280/OD315: Absorbance ratio, related to the wine’s phenolic and tannin content.
- Proline: An amino acid concentration, which can be an indicator of wine quality.

In [ ]:
# !pip install yellowbrick --user

In [ ]:
import pandas as pd
import numpy as np
from sklearn import preprocessing
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from yellowbrick.features.pca import PCADecomposition

In [ ]:
df = pd.read_csv('Wine.csv')
df.head()

### 1.	Define y and x. Standardize the predictors.
- y=class, x: the other variables
- PCA is unsupervised learning, so y is **NOT a dependent variable**. y is defined to draw a biplot.

In [ ]:
y = df['class']
x = df.drop('class', axis=1)

xscale = preprocessing.StandardScaler().fit_transform(x)  # Standardization
xscale

### 2.	Run a principal component analysis with 4 principal components. Plot the cumulative variance explained by the PCs.

In [ ]:
print(x.shape[1])
m = PCA(n_components=x.shape[1]).fit(xscale)
m = PCA().fit(xscale)  # Alternative: If n_components is omitted, all components are used

- `shape[1]`: column dimension
- `x.shape[1]`: all columns in x (=4)

In [ ]:
# Explained variance ratio
explained_variance = m.explained_variance_ratio_
explained_variance

In [ ]:
# Plot the explained variance ratio
plt.plot(range(1, len(explained_variance) + 1), explained_variance, marker='o')
plt.xlabel("Principal Component")
plt.ylabel("Explained Variance Ratio")

In [ ]:
# Cumulative variance

np.cumsum(m.explained_variance_ratio_)
np.cumsum(np.round(m.explained_variance_ratio_, decimals=3))

- `np.cumsum`: cumulative sum
- `np.round`: Round an array to the given number of decimals

In [ ]:
plt.plot(np.cumsum(m.explained_variance_ratio_), marker='o')

### 3.	Run a principal component analysis with 2 principal components. Report the two PC scores.

In [ ]:
m1=PCA(n_components=2).fit_transform(xscale)
m1[:10]  # First 10 PC scores

### 4.	Plot the samples on the two PCs.

- m1[:, 0]: all rows and 1st column
- m1[:, 1]: all rows and 2nd column

In [ ]:
# Scatter plot of the first two principal components

plt.scatter(m1[:, 0], m1[:, 1], c=y, cmap='viridis')
plt.xlabel('First Principal Component')
plt.ylabel('Second Principal Component')
plt.colorbar(label='Class')  # Add color legend

### 5.	Draw a biplot with two PCs using scaled data.

- A biplot displays both the scores of the observations and the loadings of the variables on the principal components. It provides a way to visualize the relationships between observations and variables simultaneously.
- Scores (Observations) are the projections of the original data points onto the principal component axes. In the bioplot, each point represents an observation, and their positions reflect the similarities or differences between them based on the principal components.
- Loadings (Variables) are the coefficients of the original variables in the principal component space. In the biplot, vectors or arrows are used to represent the loadings, showing the direction and magnitude of each variable's contribution to the principal components.
- The two principal components are typically plotted as the x-axis and y-axis, representing the directions of maximum variance in the data.
- The angle and length of the loading vectors indicate the contribution and correlation of each variable with the principal components. Observations that are close to each other in the biplot are similar in terms of the original variables, while observations that are far apart are dissimilar.

In [ ]:
df = pd.read_csv('Wine.csv')
y = df['class']
y_encoded = preprocessing.LabelEncoder().fit_transform(y)  # Encode y
x_scaled = pd.DataFrame(xscale, columns=x.columns)

m2=PCADecomposition(proj_features=True)
m2.fit_transform(x_scaled, y_encoded) # Use the correct y_encoded (178 elements)
m2.show()

### 6.	Draw a biplot with the two PCs using the mean value of each class.

In [ ]:
df=df.groupby(['class'], as_index=False).mean()
df

- `as_index`: controls whether the group labels (the values of 'custcat') should be used as the index of the resulting DataFrame.
    - `as_index=True` (default): the group labels will be used as the index of the resulting DataFrame.
    - `as_index=False`: the group labels will be retained as a column.

In [ ]:
y=df['class']
x=df.drop('class', axis=1)

# Standardize the grouped data and convert to DataFrame
x_scaled = pd.DataFrame(preprocessing.StandardScaler().fit_transform(x), columns=x.columns)

y=preprocessing.LabelEncoder().fit_transform(y)
m3=PCADecomposition(proj_features=True)
m3.fit_transform(x_scaled, y)
m3.show()

### 7. Interprete the plot.

- Class 1: High on most dimensions except alcalinity of ash, nonflavanoid phenols, malic acid.
- Class 2: High on alcalinity of ash, hue, od280/ed315. Low on proline, magnesium, alcohol, ash, color intensity, malic acid.
- Class 3: High on color intensity, alcohol, ash, alcalinity of ash, nonflavanoid phenols, and malic acid. Low on hue, OD280/OD315, flavanoids, proanthocyanins, and phenols.

### 8. How would you name the two axes?

- PC1 (X-axis): "Wine Composition and Color Intensity"
  - Attributes like alcohol, ash, magnesium, proline, and color intensity have strong contributions to this axis. These features are related to the overall chemical composition and visual intensity of the wine.

- PC2 (Y-axis): "Phenolic and Flavonoid Content"
  - Attributes like flavanoids, phenols, proanthocyanins, nonflavanoid phenols, and OD280/OD315 contribute significantly to this axis. These features are associated with phenolic compounds, which influence wine's taste, bitterness, and antioxidant properties.